# QICK loopback on the AntSDR E200

Fires a pulse from the QICK signal generator into the AD9361 transmitter, and
captures it on the QICK readout through the external TX -> 10 dB attenuator ->
RX loop. This is the first end-to-end test of the QICK datapath: tProcessor v2
sequencing a generator and a readout, with the averaging buffer read back over
DMA.

Run this notebook as **root** (PYNQ needs to program the PL). Order matters in
two places, both noted where they occur:

* the overlay is loaded **before** the radio is configured, because programming
  the PL re-probes the AD9361 drivers and the driver resets the sample rate --
  so anything set beforehand is lost;
* `refresh_rf()` is called **after** configuring, because `QickSocE200` reads the
  rate at construction to build the frequency plan, and a rate changed
  afterwards would leave that plan silently describing the wrong hardware.

Runs at **50 MHz** sample rate and a **6 GHz** LO.

The generator drives the transmit port directly, with no mux to select between
it and the stock ADI path. That means this overlay and the board's own
`e200_loopback_demo.ipynb` are mutually exclusive: while this bitstream is
loaded, the stock demo will not transmit. Reboot to get the base design back.

Measured at 50 MHz and 6 GHz, commanding a 10 us pulse:

| gain | arrives | length | \|IQ\| | flatness | freq error |
|------|---------|--------|---------|----------|------------|
| 0.10 | 4.48 us | 10.08 us | 11.1 | 13.7 % | +0.8 kHz |
| 0.20 | 4.48 us | 10.08 us | 22.1 | 15.2 % | +0.8 kHz |
| 0.40 | 4.48 us | 10.08 us | 44.2 | 7.4 % | +0.6 kHz |
| 0.80 | 4.48 us | 9.92 us | 87.7 | 6.5 % | +0.4 kHz |

Amplitude is linear to within 1.4 % and pulse length lands within one decimated
sample (0.16 us at 6.25 MHz). Amplitudes are about 3x lower than the same
measurement at 2.4 GHz: 6 GHz is the top of the AD9361's tuning range and the
cable and attenuator lose more there, which also explains the poorer flatness at
low gain, where the signal sits closer to the noise floor.


In [ ]:
%matplotlib inline
import sys, time
import numpy as np
import matplotlib.pyplot as plt

QICK_DIR = '/home/xilinx/qick_e200'
sys.path.insert(0, QICK_DIR)

from qick.ad9361 import QickSocE200
from qick.asm_v2 import AveragerProgramV2

# fs is deliberately not passed: it is read back from the AD9361 after the PL is
# programmed, which is the only moment the value is trustworthy.
#
# QickSocE200 loads a device tree overlay by default (the same pl.dtbo the
# board's /boot/boot.py uses) and restarts iiod afterwards. Both are required:
# programming the PL resets axi_ad9361 under the bound driver stack, and without
# the re-probe the radio is left asleep with a halved sample rate and cannot be
# woken again short of a reboot.
soc = QickSocE200(QICK_DIR + '/qick_e200.bit')
soccfg = soc
print(soc)


In [ ]:
# --- radio ON, and the frequency plan rebuilt around it ---------------------
# Two orderings conflict here, and both matter.
#
# Programming the PL re-probes the AD9361 drivers -- that is what the dtbo is
# for -- and the driver re-establishes its default 30.72 MHz in the process. So
# a sample rate set BEFORE the overlay load does not survive it.
#
# But QickSocE200 reads the rate at construction to build the frequency plan, so
# a rate set AFTER the load leaves soccfg describing a rate the hardware is not
# running. That failure is silent: every frequency and duration is simply wrong.
#
# Hence: load, configure, then refresh_rf() to rebuild the plan.
import adi

FS_MHZ = 50.0            # AD9361 sample rate; also the QICK datapath clock
LO_GHZ = 6.0             # same for TX and RX so baseband frequencies line up
BW_HZ  = int(40e6)

sdr = adi.ad9361(uri='local:')

ensm_on_entry = sdr._ctrl.attrs['ensm_mode'].value
sdr.rx_rf_bandwidth = BW_HZ
sdr.tx_rf_bandwidth = BW_HZ
sdr.sample_rate     = int(FS_MHZ * 1e6)
time.sleep(0.5)
sdr.rx_lo = int(LO_GHZ * 1e9)
sdr.tx_lo = int(LO_GHZ * 1e9)
time.sleep(0.3)

sdr.rx_enabled_channels = [0]
sdr.tx_enabled_channels = [0]

# Manual gain: an AGC would ride the level and make the result meaningless.
# These are the gains the amplitudes quoted at the top were measured at. 6 GHz
# is the top of the AD9361's range and the cable and attenuator lose more there,
# so if you want more SNR, raise rx_hardwaregain or reduce the TX attenuation --
# there is roughly 20 dB of headroom before the readout clips.
sdr.gain_control_mode_chan0 = 'manual'
sdr.rx_hardwaregain_chan0   = 20.0
sdr.tx_hardwaregain_chan0   = -20.0   # dB of attenuation, 0 = max output

for step in ('alert', 'fdd'):
    sdr._ctrl.attrs['ensm_mode'].value = step
    time.sleep(0.2)
print(f"ensm_mode: {ensm_on_entry} -> {sdr._ctrl.attrs['ensm_mode'].value}  (radio on)")

soc.refresh_rf()
fs_hw = sdr.sample_rate / 1e6
print(f"AD9361 sample rate: {fs_hw:.6f} MHz   LO: {sdr.rx_lo/1e9:.6f} GHz")
print(f"QICK  refclk_freq : {soc['refclk_freq']:.6f} MHz")
assert abs(fs_hw - soc['refclk_freq']) < 1e-6, (
    'the frequency plan does not match the hardware rate; refresh_rf() should '
    'have rebuilt it')
print('software and hardware agree on the sample rate')
print(f"readout decimated rate: {soc['readouts'][0]['f_output']:.3f} MHz")


In [ ]:
# --- enable the AD9361 transmit path ----------------------------------------
# The generator is wired straight to the transmit port: there is no mux and no
# select, so this overlay replaces the stock ADI transmit data outright and the
# board's own loopback demo will not transmit while it is loaded.
#
# What is still required is that the AD9361 has its transmit channel in DMA
# mode, which needs an active ADI transmit buffer. Without one the converter
# silently ignores the fabric data: the capture reads zero while the tProc runs,
# the shot counter increments and the descriptor reaches the generator, with no
# error anywhere. A cyclic buffer of zeros selects the mode and contributes no
# signal of its own -- the generator's samples take its place in the fifo.
sdr.tx_cyclic_buffer = True
sdr.tx(np.zeros(4096, dtype=np.complex64))
time.sleep(0.5)
print('ADI transmit path in DMA mode (cyclic zeros)')


In [ ]:
# --- the program ------------------------------------------------------------
# One constant-envelope pulse, and one readout window that opens with it. The
# readout mixes the tone down to DC, so a correct capture is a flat-topped
# envelope with a slowly rotating phase (the residual frequency error).

class LoopbackProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        # axis_readout_v2 is not driven by the tProcessor (its tproc_ch is None),
        # so QICK calls it "static": the downconversion frequency is written by
        # software here rather than sent as a readoutconfig from the program.
        # gen_ch ties the two frequency grids together so the tone lands exactly
        # at the readout's DC.
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'],
                             freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.add_pulse(ch=cfg['gen_ch'], name='probe', ro_ch=cfg['ro_ch'],
                       style='const', freq=cfg['freq'], phase=0,
                       gain=cfg['gain'], length=cfg['pulse_len'])

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name='probe', t=0)
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_t'])

f_dec = soc['readouts'][0]['f_output']        # decimated sample rate, MHz
config = dict(
    gen_ch    = 0,
    ro_ch     = 0,
    freq      = 1.0,     # MHz of baseband offset; DDS range is +/- fs/2
    gain      = 0.5,
    pulse_len = 10.0,    # us
    ro_len    = 30.0,    # us -> about %d decimated samples
    trig_t    = 0.0,
)
print(f"decimated rate {f_dec:.3f} MHz -> {config['ro_len']*f_dec:.0f} samples "
      f"in a {config['ro_len']:.0f} us window")


In [ ]:
# --- fire it and capture ----------------------------------------------------
prog = LoopbackProgram(soccfg, reps=1, final_delay=1.0, cfg=config)
iq = prog.acquire_decimated(soc, rounds=10, progress=False)

# acquire_decimated returns one array per readout, shaped (nsamples, 2)
d = iq[0]
i, q = d[:, 0], d[:, 1]
t = np.arange(len(i)) / f_dec        # us
mag = np.abs(i + 1j*q)

# The pulse does not begin at t=0: there is roughly 6 us of round-trip latency.
# Detect it rather than assuming, or every measurement is taken over dead time.
thr = 0.3 * mag.max()
on = np.flatnonzero(mag > thr)
print(f"captured {len(i)} decimated samples over {t[-1]:.1f} us")
print(f"peak |IQ| = {mag.max():.1f} ADU at t = {t[np.argmax(mag)]:.2f} us")
if len(on) >= 4:
    print(f"pulse arrives at {t[on[0]]:.2f} us, length {t[on[-1]] - t[on[0]]:.2f} us"
          f" (commanded {config['pulse_len']:.1f} us)")

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(t, i, '.-', label='I')
ax[0].plot(t, q, '.-', label='Q')
ax[0].set_ylabel('ADU'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title('QICK readout, decimated: pulse through the external loopback')
ax[1].plot(t, mag, '.-', color='k')
if len(on) >= 4:
    ax[1].axvspan(t[on[0]], t[on[-1]], color='tab:orange', alpha=.15,
                  label='detected pulse')
ax[1].set_xlabel('time (us)'); ax[1].set_ylabel('|IQ| (ADU)')
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()


In [ ]:
# --- is it the right signal? ------------------------------------------------
# Inside the pulse the readout should have mixed the tone to near DC, so the
# residual tone in the decimated data tells us the frequency error. Anything
# near the full decimated bandwidth would mean the readout is demodulating at
# the wrong frequency, or the pulse is not the thing we are seeing.

# Measure inside the detected pulse, skipping its edges.
seg = (i + 1j*q)[on[0] + 2 : on[-1] - 1] if len(on) >= 6 else np.array([])
if len(seg) < 8:
    print('pulse window too short to estimate frequency; lengthen pulse_len')
else:
    win = np.hanning(len(seg))
    sp = np.fft.fftshift(np.fft.fft(seg * win))
    fx = np.fft.fftshift(np.fft.fftfreq(len(seg), d=1.0/f_dec))
    f_err = fx[np.argmax(np.abs(sp))]
    # phase slope is a finer estimate than the FFT bin at this record length
    ph = np.unwrap(np.angle(seg))
    slope = np.polyfit(np.arange(len(seg))/f_dec, ph, 1)[0] / (2*np.pi)
    print(f"residual tone: {f_err*1e3:+.1f} kHz (FFT bin, {f_dec/len(seg)*1e3:.1f} kHz resolution)")
    print(f"               {slope*1e3:+.3f} kHz (phase slope)")
    print(f"in-pulse |IQ|: mean {np.abs(seg).mean():.1f}, "
          f"ripple {100*(np.abs(seg).max()-np.abs(seg).min())/np.abs(seg).mean():.1f} %")

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(fx*1e3, 20*np.log10(np.abs(sp)/np.abs(sp).max() + 1e-12), '.-')
    ax.set_xlabel('offset from readout frequency (kHz)')
    ax.set_ylabel('dB (rel. peak)')
    ax.set_title('Spectrum inside the pulse: a correct demodulation sits at DC')
    ax.grid(alpha=.3)
    plt.tight_layout()


In [ ]:
# --- put everything back ----------------------------------------------------
# soc.standby() parks the radio in ALERT and takes the transmitter to maximum
# attenuation. ALERT keeps the BBPLL and the data interface running -- so the
# clock the QICK datapath runs on survives -- while powering down the signal
# paths, where the heat comes from: measured 74.6 C in FDD settling to about
# 44 C in ALERT, and 7 C off the Zynq die.
#
# Not SLEEP: that stops the LVDS DATA_CLK, and an AXI access with no clock
# behind it hangs the interconnect hard enough to need a power cycle.
#
# QickSocE200 also registers this to run when the process exits, but atexit
# fires when the kernel stops rather than when this cell finishes, so call it
# here to hand the radio back now.
try:
    sdr.tx_destroy_buffer()
except Exception:
    pass

print('ensm_mode now:', soc.standby(), '(radio parked)')
del sdr
print()
print('Note the PL still holds the QICK overlay, not the boot-time base design.')
print('Reboot to return the board to its stock state.')
